# 도구 오류와 ToolMessage

도구 실행 오류를 그래프 밖의 예외로 끝내지 않고 모델이 읽을 ToolMessage로 바꾸려면 무엇을 설정해야 할까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 순서대로 실행합니다. API 키 값은 출력하지 않습니다.

## 패키지와 환경 준비

고정한 패키지 버전과 API 키 존재 여부만 확인합니다. 키 값 자체는 출력하지 않습니다.

## 코드를 읽기 전에

도구 함수가 `ValueError`를 일으키면 기본 실행은 예외로 중단될 수 있습니다. `handle_tool_errors=False`인 ToolNode는 이 예외를 밖으로 전파해 실패 위치를 그대로 보여 줍니다. 오류 처리 함수를 지정한 ToolNode는 허용한 예외를 문자열로 바꾸고, 원래 요청 ID를 가진 ToolMessage에 담아 상태에 추가합니다.

ToolMessage가 생겼다는 사실은 도구가 성공했다는 뜻이 아닙니다. `content`를 읽어 정상 결과와 처리된 오류를 구분해야 하며, 모델은 그 메시지를 다음 입력으로 받아 사용자 설명이나 다른 행동을 선택합니다.

In [1]:
# 패키지와 환경 준비에 필요한 연산을 실행합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / '.env').is_file():
        load_dotenv(folder / '.env')
        break
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY가 필요합니다.'
print('langgraph', version('langgraph'))
print('langchain', version('langchain'))
print('langchain-openai', version('langchain-openai'))
print('API 키 존재 여부 확인 완료')

Note: you may need to restart the kernel to use updated packages.
langgraph 1.2.11
langchain 1.4.0
langchain-openai 1.6.0
API 키 존재 여부 확인 완료


## 모델과 비용 함수 준비

실습에 사용할 모델과 토큰 사용량으로 추정 비용을 계산하는 함수를 준비합니다.

In [2]:
# 모델과 비용 함수 준비에 필요한 연산을 실행합니다.
from time import perf_counter
from typing import Any
from langchain.messages import AIMessage
from langchain_core.language_models import BaseChatModel
from langchain.chat_models import init_chat_model

MODEL_NAME = 'gpt-5.4-mini'
INPUT_USD_PER_MTOK = 0.75
OUTPUT_USD_PER_MTOK = 4.50

def usage_of(message: AIMessage) -> dict:
    usage = message.usage_metadata
    assert usage is not None, 'usage_metadata가 없어 비용을 계산할 수 없습니다.'
    assert 'input_tokens' in usage and 'output_tokens' in usage
    input_tokens = usage['input_tokens']
    output_tokens = usage['output_tokens']
    cost = input_tokens / 1_000_000 * INPUT_USD_PER_MTOK
    cost += output_tokens / 1_000_000 * OUTPUT_USD_PER_MTOK
    return {'input_tokens': input_tokens, 'output_tokens': output_tokens, 'estimated_cost_usd': cost}

def make_model() -> BaseChatModel:
    return init_chat_model(f'openai:{MODEL_NAME}', reasoning_effort='none')

## 읽기 전용 규정 도구 준비

지원하지 않는 주제를 받으면 ValueError를 내는 lookup_policy 도구를 정의합니다.

In [3]:
# 읽기 전용 규정 도구 준비에 필요한 연산을 실행합니다.
from langchain.tools import tool

POLICIES = {
    '대출': '일반 도서는 1인 5권까지 14일 동안 대출할 수 있습니다.',
    '운영': '평일은 09:00~20:00, 토요일은 10:00~17:00에 운영합니다.',
    '연체': '연체한 일수만큼 새 대출이 제한됩니다.',
}

@tool
def lookup_policy(topic: str) -> str:
    """교육센터 자료실 규정을 주제어로 조회합니다. topic은 대출, 운영, 연체 중 하나입니다."""
    if topic not in POLICIES:
        raise ValueError(f'지원하지 않는 주제: {topic}')
    return POLICIES[topic]

TOOLS = [lookup_policy]

## 같은 실패 도구 호출 만들기

두 비교에서 같은 요청을 쓰도록 topic이 '주차'인 AIMessage를 고정합니다.

In [4]:
# 같은 실패 도구 호출 만들기에 필요한 연산을 실행합니다.
from langchain.messages import AIMessage

bad_call = AIMessage(content='', tool_calls=[{
    'name': 'lookup_policy', 'args': {'topic': '주차'}, 'id': 'call_bad_1', 'type': 'tool_call'
}])
print(bad_call.tool_calls)

[{'name': 'lookup_policy', 'args': {'topic': '주차'}, 'id': 'call_bad_1', 'type': 'tool_call'}]


## Before: 예외가 그래프 밖으로 전파됨

handle_tool_errors=False인 ToolNode에서 ValueError가 호출자에게 전파되는지 확인합니다.

In [5]:
# Before: 예외가 그래프 밖으로 전파됨에 필요한 연산을 실행합니다.
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode

strict_builder = StateGraph(MessagesState)
strict_builder.add_node('tools', ToolNode(TOOLS, handle_tool_errors=False))
strict_builder.add_edge(START, 'tools')
strict_builder.add_edge('tools', END)
strict_graph = strict_builder.compile()
try:
    strict_graph.invoke({'messages': [bad_call]})
except ValueError as error:
    before_error = f'{type(error).__name__}: {error}'
    print(before_error)
assert before_error.startswith('ValueError:')

ValueError: 지원하지 않는 주제: 주차


## After: 오류 처리 함수 정의

ValueError만 처리해 지원 주제를 안내하는 ToolMessage로 바꾸는 함수를 정의합니다.

In [6]:
# After: 오류 처리 함수 정의에 필요한 연산을 실행합니다.
def handle_policy_error(error: ValueError) -> str:
    return f'규정 조회 실패: {error}. topic은 대출, 운영, 연체 중 하나를 사용하세요.'

safe_node = ToolNode(TOOLS, handle_tool_errors=handle_policy_error)
safe_builder = StateGraph(MessagesState)
safe_builder.add_node('tools', safe_node)
safe_builder.add_edge(START, 'tools')
safe_builder.add_edge('tools', END)
safe_graph = safe_builder.compile()
after_result = safe_graph.invoke({'messages': [bad_call]})
tool_error = after_result['messages'][-1]
print(type(tool_error).__name__)
print('tool_call_id:', tool_error.tool_call_id)
print('content:', tool_error.content)
assert tool_error.tool_call_id == 'call_bad_1'

ToolMessage
tool_call_id: call_bad_1
content: 규정 조회 실패: 지원하지 않는 주제: 주차. topic은 대출, 운영, 연체 중 하나를 사용하세요.


## After: 실제 모델이 오류 메시지 읽기

오류 ToolMessage를 실제 모델에 전달해 지원 범위를 설명하는 답과 사용량을 확인합니다.

In [7]:
# After: 실제 모델이 오류 메시지 읽기에 필요한 연산을 실행합니다.
from langchain.messages import HumanMessage

model = make_model()
conversation = [
    HumanMessage('자료실 주차 규정을 알려 주세요.'),
    bad_call,
    tool_error,
]
started = perf_counter()
reply = model.invoke(conversation)
elapsed = perf_counter() - started
usage = usage_of(reply)
print('answer:', reply.text)
print('usage:', usage)
print('latency_sec:', round(elapsed, 3))
assert reply.text and usage['estimated_cost_usd'] < 1.0

answer: 죄송하지만 현재 조회 가능한 자료실 규정은 **대출, 운영, 연체**뿐이라서 **주차 규정**은 확인할 수 없습니다.

원하시면 아래 중 하나를 알려 주세요.
- **대출 규정**
- **운영 규정**
- **연체 규정**

또는 주차에 대해 확인해야 할 항목이 있다면, 제가 질문 문구를 정리해 드릴게요.
usage: {'input_tokens': 73, 'output_tokens': 97, 'estimated_cost_usd': 0.0004912499999999999}
latency_sec: 1.356


## 다른 상황에 적용하기

재고 ID가 없다는 업무 오류는 ‘존재하지 않음’ 메시지로 바꾸어 모델이 다른 ID를 묻게 할 수 있습니다. 인증 실패나 데이터베이스 손상처럼 복구 정책이 다른 오류까지 같은 문장으로 바꾸면 운영 진단이 어려워집니다. 처리할 예외 유형, 사용자에게 공개할 내용, 기록할 원인을 나누어 설계합니다.